# Scientist #8: Robert Hooke — Dressed MPFB2 Human Cast + Crisp 3D Typography + Cycles Turbo

**@zackdfilms-Style Realistic Anatomical Human Cast (MPFB2 / MakeHuman for Blender 4.2) + Zero-Compromise Cycles Turbo (~6-8 Min on Kaggle `GPU T4 x2`)!**
- **Tailored 3D 1665 Costumes & Realistic Cast**: Robert Hooke (3D solidified forest-emerald velvet coat, pleated lace cravat, lace cuffs, brass buttons, tapered NURBS chestnut waves, flush eyebrows, SSS skin) & Rival Watchmaker (3D burgundy velvet coat, powdered silver wig, flush curled mustache) with natural verified hand/arm acting & exact Kokoro jaw lip-sync!
- **Razor-Sharp 3D Typography & Guaranteed Subtitles**: Bundled `DejaVuSans-Bold.ttf` powers BOTH Blender 3D text (`ceiiinosssttuv`, `F = - k x`, luminous gold `HOOKE'S LAW`) AND FFmpeg `libass` boxless karaoke subtitles (`fontsdir=subtitles`)!
- **Zero-Overlap Sightlines**: Every Beat stages characters cleanly BESIDE each instrument so 100% of faces AND scientific apparatus shine unobstructed!

### Quick Kaggle Setup:
1. Right panel -> **Session options** -> **Accelerator: `GPU T4 x2`** & **Internet: `ON`**.
2. Run **Cell 1 -> Cell 2 -> Cell 3** (~18s build + ~6s **6-Beat Visual Preview Grid** showing dressed Robert Hooke, Rival & crisp 3D text!).
3. Run **Cell 4 -> Cell 5** (~6-8 min) -> download **`/kaggle/working/robert_hooke_elasticity_short.mp4`**!

In [ ]:
# 1. Verify NVIDIA GPU(s) & Clone Complete Repo (includes audio + lip-sync + bundled DejaVuSans-Bold.ttf + subtitles)
!nvidia-smi
!rm -rf hooke-elasticity-micrographia-3d
!git clone https://github.com/ssghtc002-sudo/hooke-elasticity-micrographia-3d.git
%cd hooke-elasticity-micrographia-3d
!ls -lh audio/output/master_audio.wav audio/output/voice_manifest.json subtitles/subtitles.ass subtitles/fonts/DejaVuSans-Bold.ttf

In [ ]:
# 2. Install Blender 4.2.3 LTS + MPFB2 Realistic Human Generator Extension (~35 sec total, cached in /tmp)
import os
if not os.path.exists('/tmp/blender/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /tmp/blender && tar -xf /tmp/blender.tar.xz -C /tmp/blender --strip-components=1

if not os.path.exists('/tmp/mpfb.zip'):
    !git clone --depth 1 -q https://github.com/makehumancommunity/mpfb2.git /tmp/mpfb2
    !(cd /tmp/mpfb2/src && zip -qr /tmp/mpfb.zip mpfb)
    !/tmp/blender/blender -b --command extension install-file -r user_default -e /tmp/mpfb.zip

!/tmp/blender/blender --version

In [ ]:
# 3. Fast 6-Beat Visual QA Preview (~18s scene build + ~6s pure render!) — Inspect Dressed Characters & Crisp 3D Text!
import glob, os, base64
from IPython.display import display, HTML

!rm -rf qa_frames
!CUDA_VISIBLE_DEVICES=0 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_hooke_scene.py -- --qa-only --fps 24 --samples 10 --scale 75

imgs = sorted(glob.glob('qa_frames/qa_beat*.jpg'))
cards = []
for p in imgs:
    b64 = base64.b64encode(open(p, 'rb').read()).decode()
    cards.append(f'<div style="display:inline-block;margin:6px;text-align:center;"><img src="data:image/jpeg;base64,{b64}" width="210"/><br><small>{os.path.basename(p)}</small></div>')
display(HTML(''.join(cards)))

In [ ]:
# 4. Parallel Dual-GPU Cycles Turbo Render (720 frames @ 24fps, 10 spp OIDN, 75% -> 1080p Lanczos -> ~6-8 min on T4 x2!)
import subprocess, time, glob

TOTAL = 720
!rm -rf render_frames
gpu_count = int(subprocess.check_output('nvidia-smi --list-gpus | wc -l', shell=True).decode().strip())
print(f'Detected {gpu_count} NVIDIA GPU(s)! Starting Cycles Turbo production render ({TOTAL} frames @ 24fps)...')
t0 = time.time()

if gpu_count >= 2:
    p0 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=0 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_hooke_scene.py -- --start-frame 1 --end-frame 360 --fps 24 --samples 10 --scale 75',
        shell=True, stdout=open('/tmp/gpu0.log', 'w'), stderr=subprocess.STDOUT
    )
    p1 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=1 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_hooke_scene.py -- --start-frame 361 --end-frame 720 --fps 24 --samples 10 --scale 75',
        shell=True, stdout=open('/tmp/gpu1.log', 'w'), stderr=subprocess.STDOUT
    )
    while p0.poll() is None or p1.poll() is None:
        done = len(glob.glob('render_frames/frame_*.jpg'))
        elapsed = time.time() - t0
        rate = done / max(1.0, elapsed)
        eta = (TOTAL - done) / max(0.01, rate) if done > 2 else 0
        print(f'\r[Dual-T4 Turbo] {done}/{TOTAL} frames ({done*100/TOTAL:.1f}%) | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m   ', end='', flush=True)
        time.sleep(3)
    print()
    if p0.returncode != 0 or p1.returncode != 0:
        print('--- GPU 0 LOG TAIL ---\n' + ''.join(open('/tmp/gpu0.log').readlines()[-25:]))
        print('--- GPU 1 LOG TAIL ---\n' + ''.join(open('/tmp/gpu1.log').readlines()[-25:]))
        raise RuntimeError(f'Worker error: p0={p0.returncode} p1={p1.returncode}')
else:
    !OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_hooke_scene.py -- --start-frame 1 --end-frame 720 --fps 24 --samples 10 --scale 75

rendered = len(glob.glob('render_frames/frame_*.jpg'))
print(f'Render complete: {rendered}/{TOTAL} frames in {(time.time()-t0)/60:.1f} min!')

In [ ]:
# 5. Lanczos Crisp 1080x1920 Upscale + Mastered 30s Kokoro Audio + Bundled-Font Boxless Subtitles -> Final MP4!
!mkdir -p output
!ffmpeg -y -framerate 24 -i render_frames/frame_%04d.jpg -i audio/output/master_audio.wav \
  -vf "scale=1080:1920:flags=lanczos,unsharp=3:3:0.35:3:3:0.0,ass=subtitles/subtitles.ass:fontsdir=subtitles/fonts" \
  -c:v libx264 -profile:v high -crf 15 -pix_fmt yuv420p \
  -c:a aac -b:a 192k -shortest -movflags +faststart output/robert_hooke_elasticity_short.mp4 -loglevel warning

# Copy directly to /kaggle/working/ so it appears immediately in Kaggle Output panel for 1-tap download!
import os, shutil
out_mp4 = 'output/robert_hooke_elasticity_short.mp4'
if os.path.isdir('/kaggle/working'):
    shutil.copy2(out_mp4, '/kaggle/working/robert_hooke_elasticity_short.mp4')
    print('Copied to /kaggle/working/robert_hooke_elasticity_short.mp4 — Ready in Kaggle Output Panel!')

!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames,duration -of compact output/robert_hooke_elasticity_short.mp4
!ls -lh output/robert_hooke_elasticity_short.mp4

try:
    from google.colab import files
    files.download(out_mp4)
except ImportError:
    from IPython.display import Video, display
    display(Video(out_mp4, embed=False, width=320))